### Building a RAG System with LangChain and FAISS 
Introduction to RAG (Retrieval-Augmented Generation)
RAG combines the power of retrieval systems with generative AI models. Instead of relying solely on the model's training data, RAG:

1. Retrieves relevant documents from a knowledge base
2. Uses these documents as context for the LLM
3. Generates responses based on both the retrieved context and the model's knowledge

### FAISS 
https://github.com/facebookresearch/faiss

FAISS is a library for efficient similarity search and clustering of dense vectors.

Key advantages:
1. Extremely fast similarity search
2. Memory efficient
3. Supports GPU acceleration
4. Can handle millions of vectors

How it works:
- Indexes vectors for fast nearest neighbor search
- Returns most similar vectors based on distance metrics


In [2]:
## load libraries
import os
from dotenv import load_dotenv
import numpy as np

import warnings
warnings.filterwarnings('ignore')

## LangChain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

## LangChain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter

# from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

## Load environment variables
load_dotenv()

True

### Data Ingestion And Processing


In [3]:
## Create some sample documents
sample_documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
        """,
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
    ),
    Document(
        page_content="""
        Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
        """,
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"}
    ),
    Document(
        page_content="""
        Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
        """,
        metadata={"source": "Deep Learning", "page": 1, "topic": "DL"}
    ),
    Document(
        page_content="""
        Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sentiment analysis, and text summarization.
        """,
        metadata={"source": "NLP Overview", "page": 1, "topic": "NLP"}
    )
]

print(sample_documents)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='\n        Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.\n        '), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='\n        Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.\n        '), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='\n        Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolu

In [4]:
## text splitting
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len,
    separators = [" "]
)

## split the documents into chunks
chunks = text_splitter.split_documents(sample_documents)

print(f"Created {len(chunks)} chunks from {len(sample_documents)} documents")
print("\nExample chunk:")
print(f"Content: {chunks[0].page_content}")
print(f"Metadata: {chunks[0].metadata}")

Created 4 chunks from 4 documents

Example chunk:
Content: Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
Metadata: {'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}


In [6]:
# Initialize OpenAI embeddings with the latest model
# embeddings=OpenAIEmbeddings(
#     model="text-embedding-3-small",
#     dimensions=1536
# )

## Initialize a simple HuggingFace embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

## Example: create a embedding for a single text
sample_text = "What is machine learning"
sample_embedding = embeddings.embed_query(sample_text)
print(sample_embedding)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[-0.0290356632322073, 0.0075597334653139114, 0.04076480492949486, 0.030535517260432243, 0.0517570786178112, -0.017347536981105804, -0.03094184398651123, -0.06556293368339539, -0.033060770481824875, -0.009561242535710335, -0.09131379425525665, 0.04738360643386841, 0.02773955650627613, -0.06366030871868134, -0.0650475025177002, 0.042002636939287186, -0.040219034999608994, 0.028005044907331467, -0.02804984524846077, -0.053762856870889664, -0.005428346339613199, 0.0060848030261695385, -0.07086580246686935, 0.02593003213405609, 0.010079479776322842, 0.02629343792796135, 0.03857455030083656, 0.022752385586500168, -0.016700467094779015, 0.00941805075854063, 0.01637602411210537, -0.059243232011795044, -0.016352448612451553, 0.045799098908901215, -0.06120177358388901, 0.06117099151015282, -0.013366815634071827, -0.000311074050841853, 0.03935336321592331, -0.04183916747570038, -0.03927678242325783, -0.1010735034942627, -0.0060851192101836205, -0.011100593954324722, 0.12034150958061218, 0.1039781

In [7]:
texts = ["AI", "Machine learning", "Deep Learning", "Neural Network"]
batch_embeddings = embeddings.embed_documents(texts)
print(batch_embeddings[0])

[-0.03653926029801369, -0.015164357610046864, 0.016432464122772217, 0.010568826459348202, 0.006010559853166342, -0.018473288044333458, 0.08546527475118637, 0.02096845768392086, 0.027815353125333786, 0.012431694194674492, -0.02937648445367813, -0.031135285273194313, 0.03491251543164253, -0.018150851130485535, -0.06498481333255768, 0.0516824871301651, -0.019606219604611397, -0.015734203159809113, -0.13371676206588745, -0.09645991772413254, -0.02547178603708744, -0.0014895843341946602, -0.006349374074488878, -0.02582065388560295, -0.02737179957330227, 0.12268992513418198, -0.007792469579726458, -0.03852269425988197, 0.014383504167199135, -0.09218426793813705, 0.008695731870830059, 0.00261337636038661, 0.09103471785783768, -0.030313612893223763, -0.09604638814926147, 0.022289087995886803, -0.09024307876825333, -0.032947368919849396, 0.0715833380818367, -0.008893106132745743, -0.025708934292197227, -0.0791396051645279, 0.014530384913086891, -0.07420430332422256, 0.08045009523630142, 0.07804

In [8]:
## Compare Embedding using cosine similarity
def compare_embeddings(text1:str, text2:str):
    """Compare semantic simialrity of 2 texts usign embeddings"""

    emb1=np.array(embeddings.embed_query(text1))
    emb2=np.array(embeddings.embed_query(text2))

    ## Calculate the similarity score
    similarity=np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
    return similarity

## Test semantic similarity
print("\nSemantic Similarity Examples:")
print(f"'AI' vs 'Artificial Intelligence': {compare_embeddings('AI', 'Artificial Intelligence'):.3f}")
print(f"'Machine Learning' vs 'ML': {compare_embeddings('Machine Learning', 'ML'):.3f}")
print(f"'AI' vs 'Pizza': {compare_embeddings('AI', 'Pizza'):.3f}")


Semantic Similarity Examples:
'AI' vs 'Artificial Intelligence': 0.791
'Machine Learning' vs 'ML': 0.373
'AI' vs 'Pizza': 0.257


### Create FAISS Vector Store

In [9]:
## Create FAISS vector store
vectorstore = FAISS.from_documents(
    documents = chunks,
    embedding = embeddings
)

print(f"Vector store created with {vectorstore.index.ntotal} vectors")

## Save vector tore for later use
vectorstore.save_local("faiss_index")
print("Vector store saved to 'faiss_index' directory")

vectorstore

Vector store created with 4 vectors
Vector store saved to 'faiss_index' directory


In [ ]:
## load vector store
loaded_vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization = True
)

print(f"Loaded vector store contains {loaded_vectorstore.index.ntotal} vectors")

## Similarity Search 
query = "What is deep learning?"

results = vectorstore.similarity_search(query,k=3)
print(results)

Loaded vector store contains 4 vectors
[Document(id='4c2a344e-38e6-46a2-a1b3-07a3fc5af11c', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recognition.'), Document(id='350df6be-3095-4338-a8d5-dbf2d9aff24d', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'), Document(id='1fd6f437-b097-48c6-8f2c-e87ae8260bcf', metadata={'source': 'NLP Overview', 'page': 1, 'topic': 'NLP'}, page_content='Natural Language Processing (NLP) is a branch of AI tha

In [11]:
print(f"Query: {query}\n")
print("Top 3 similar chunks:")
for i, doc in enumerate(results):
    print(f"\n{i+1}. Source: {doc.metadata['source']}")
    print(f"   Content: {doc.page_content[:200]}...")

Query: What is deep learning?

Top 3 similar chunks:

1. Source: Deep Learning
   Content: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning ...

2. Source: ML Basics
   Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised...

3. Source: NLP Overview
   Content: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
      ...


In [12]:
## Similarity Search with score
results_with_scores = vectorstore.similarity_search_with_score(query,k=3)

print("\n\nSimilarity search with scores:")
for doc, score in results_with_scores:
    print(f"\nScore: {score:.3f}")
    print(f"Source: {doc.metadata['source']}")
    print(f"Content preview: {doc.page_content[:100]}...")



Similarity search with scores:

Score: 0.394
Source: Deep Learning
Content preview: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses m...

Score: 1.007
Source: ML Basics
Content preview: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being...

Score: 1.112
Source: NLP Overview
Content preview: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
...


In [ ]:
## Search with metadata filtering
filter_dict = {"topic":"ML"}
filtered_results = vectorstore.similarity_search(query, k = 3, filter = filter_dict)

print(filtered_results)

[Document(id='350df6be-3095-4338-a8d5-dbf2d9aff24d', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.')]


### Build RAG Chain With LCEL 

In [15]:
## GROQ LLM
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

llm = init_chat_model("groq:llama-3.3-70b-versatile")
llm.invoke("Hi")

AIMessage(content="It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 36, 'total_tokens': 59, 'completion_time': 0.050867329, 'completion_tokens_details': None, 'prompt_time': 0.001728791, 'prompt_tokens_details': None, 'queue_time': 0.162346118, 'total_time': 0.05259612}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fc6b9-092a-7f91-9eeb-37c865cabcae-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 23, 'total_tokens': 59})

In [16]:
# 1. Simple RAG Chain with LCEL
simple_prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:
    Context: {context}
    Question: {question}
    Answer:"""
)

## Basic retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002A6C41AD7F0>, search_kwargs={'k': 3})

In [17]:
from typing import List
## Format documents for the prompt
def format_docs(docs: List[Document]) -> str:
    """Format documents for insertion into prompt"""
    formatted = []
    
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'Unknown')
        formatted.append(f"Document {i+1} (Source: {source}):\n{doc.page_content}")
    
    return "\n\n".join(formatted)

## Create the RAG simple chain with LCEL
simple_rag_chain=(
    {"context":retriever | format_docs,"question":RunnablePassthrough() }
    | simple_prompt
    | llm
    | StrOutputParser()
)

simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002A6C41AD7F0>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n    Context: {context}\n    Question: {question}\n    Answer:'), additional_kwargs={})])
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions

In [18]:
## Conversational RAG Chain
conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the provided context to answer questions."),
    ("placeholder", "{chat_history}"),
    ("human", "Context: {context}\n\nQuestion: {input}"),
])

def create_conversational_rag():
    """Create a conversational RAG chain with memory"""
    return (
        RunnablePassthrough.assign(
            context=lambda x: format_docs(retriever.invoke(x["input"]))
        )
        | conversational_prompt
        | llm
        | StrOutputParser()
    )

conversational_rag = create_conversational_rag()
conversational_rag

RunnableAssign(mapper={
  context: RunnableLambda(lambda x: format_docs(retriever.invoke(x['input'])))
})
| ChatPromptTemplate(input_variables=['context', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag=

In [19]:
## streaming RAG chain
streaming_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | simple_prompt
    | llm
)

print("Modern RAG chains created successfully!")
print("Available chains:")
print("- simple_rag_chain: Basic Q&A")
print("- conversational_rag: Maintains conversation history")
print("- streaming_rag_chain: Supports token streaming")

Modern RAG chains created successfully!
Available chains:
- simple_rag_chain: Basic Q&A
- conversational_rag: Maintains conversation history
- streaming_rag_chain: Supports token streaming


In [21]:
## Test function for different chain types
def test_rag_chains(question: str):
    """Test all RAG chain variants"""
    print(f"Question: {question}")
    print("=" * 80)
    
    ## 1. Simple RAG
    print("\n1. Simple RAG Chain:")
    answer = simple_rag_chain.invoke(question)
    print(f"Answer: {answer}")

    ## 2. Streaming RAG
    print("\n2. Streaming RAG:")
    print("Answer: ", end="", flush=True)
    for chunk in streaming_rag_chain.stream(question):
        print(chunk.content, end="", flush=True)
    
    print()

test_rag_chains("What is the difference between AI and machine learning")

Question: What is the difference between AI and machine learning

1. Simple RAG Chain:
Answer: According to the provided context, the difference between AI and Machine Learning is as follows:

Artificial Intelligence (AI) is the simulation of human intelligence in machines, designed to think like humans and mimic their actions. On the other hand, Machine Learning (ML) is a subset of AI that enables systems to learn from data, finding patterns in data without being explicitly programmed.

In other words, AI is the broader field that encompasses Machine Learning, which is a specific technique used to achieve AI's goals. While AI focuses on simulating human intelligence, Machine Learning is a key approach to achieving this simulation by enabling systems to learn from data.

2. Streaming RAG:
Answer: According to the context, the difference between AI and machine learning is that:

Artificial Intelligence (AI) is the simulation of human intelligence in machines, designed to think like huma

In [22]:
## Test with multiple questions
test_questions = [
    "What is the difference between AI and Machine Learning?",
    "Explain deep learning in simple terms",
    "How does NLP work?"
]

for question in test_questions:
    print("\n" + "=" * 80 + "\n")
    test_rag_chains(question)



Question: What is the difference between AI and Machine Learning?

1. Simple RAG Chain:
Answer: According to the provided context, Artificial Intelligence (AI) refers to the simulation of human intelligence in machines, where systems are designed to think like humans and mimic their actions. On the other hand, Machine Learning (ML) is a subset of AI that enables systems to learn from data, finding patterns without being explicitly programmed.

In other words, AI is a broader concept that encompasses the idea of creating intelligent machines, while Machine Learning is a specific approach within AI that focuses on enabling systems to learn from data. This means that all Machine Learning is AI, but not all AI is Machine Learning.

2. Streaming RAG:
Answer: According to the context, the difference between AI and Machine Learning is that Artificial Intelligence (AI) is the simulation of human intelligence in machines, designed to think like humans and mimic their actions, whereas Machine 

In [23]:
## Conversational example
print("\n3. Conversational RAG Example:")
chat_history = []

# First question
q1 = "What is machine learning?"
a1 = conversational_rag.invoke({
    "input": q1,
    "chat_history": chat_history
})

print(f"Q1: {q1}")
print(f"A1: {a1}")

## Update history
chat_history.extend([
    HumanMessage(content=q1),
    AIMessage(content=a1)
])


3. Conversational RAG Example:
Q1: What is machine learning?
A1: According to Document 1 (Source: ML Basics), Machine Learning is a subset of AI that enables systems to learn from data, instead of being explicitly programmed, by finding patterns in data.


In [24]:
## Follow-up question
q2 = "How is it different from traditional programming?"
a2 = conversational_rag.invoke({
    "input": q2,
    "chat_history": chat_history
})
print(f"\nQ2: {q2}")
print(f"A2: {a2}")


Q2: How is it different from traditional programming?
A2: According to Document 1 (Source: ML Basics), Machine Learning is different from traditional programming in that it enables systems to learn from data instead of being explicitly programmed. This means that with traditional programming, a system is given a set of rules and instructions to follow, whereas with Machine Learning, the system is able to find patterns in data and learn from it on its own.
